# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 470,000 new records
write_files: 470,000 records -> /home/jovyan/data/lake/logs/run-20261121-133031-a64381

Lake check — topic 'logs' (4 partitions)   2026-11-21 13:31:13
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 308,715)     114,715           0          0     194,000
        1           [0, 338,925)     124,925           0          0     214,000
        2           [0, 321,436)     105,436           0          0     216,000
        3           [0, 346,924)     124,924           0          0     222,000
-------------------------------------------------------------------------------
    total      1,316,000 records     470,000           0          0     846,000

files under /home/jovyan/data/lake: 4 data files, 26.2 MB, 6.56 MB per file

{'topic': 1316000,
 'lake': 470000,
 'dup': 0,
 'miss': 0,
 'lag': 846000,
 'files': 4,
 'mb': 26.230179,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 1,319,345 new records
write_files: 1,319,345 records -> /home/jovyan/data/lake/logs/run-20261121-133113-816c7c

Lake check — topic 'logs' (4 partitions)   2026-11-21 13:31:30
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 386,715)     423,430     114,715          0      78,000
        1           [0, 426,490)     465,195     124,925          0      86,220
        2           [0, 409,436)     426,872     105,436          0      88,000
        3           [0, 432,924)     473,848     124,924          0      84,000
-------------------------------------------------------------------------------
    total      1,655,565 records   1,789,345     470,000          0     336,220

files under /home/jovyan/data/lake: 8 data files, 99.8 MB, 12.47 MB per file on average
check took 0.1 s

Exac

{'topic': 1655565,
 'lake': 1789345,
 'dup': 470000,
 'miss': 0,
 'lag': 336220,
 'files': 8,
 'mb': 99.772139,
 'ok': False,
 'problems': ['470,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [5]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    
    # 2. read what is new
    new = read_new(start)
    
    # 3. write it
    run = write_files(new, STAGING)
    
    # 4. remember where you stopped (if your architecture needs it)
    publish(run)

reset()
run_once()
check()

reset: lake, staging and state deleted
read_new: 1,780,000 new records
write_files: 1,780,000 records -> /home/jovyan/data/staging/run-20261121-133136-25fb63
publish: run-20261121-133136-25fb63 is now in the lake

Lake check — topic 'logs' (4 partitions)   2026-11-21 13:31:56
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 468,715)     426,715           0          0      42,000
        1           [0, 508,925)     448,925           0          0      60,000
        2           [0, 521,436)     447,436           0          0      74,000
        3           [0, 500,924)     456,924           0          0      44,000
-------------------------------------------------------------------------------
    total      2,000,000 records   1,780,000           0          0     220,000

files under /home/

{'topic': 2000000,
 'lake': 1780000,
 'dup': 0,
 'miss': 0,
 'lag': 220000,
 'files': 4,
 'mb': 99.201718,
 'ok': True,
 'problems': []}

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [6]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,000,000           0        0           0  19.4s  PASS
2 new records arrive, run again         2,100,000           0        0           0   3.6s  PASS
3 nothing new, run again                2,100,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,100,000           0        0     100,000         PASS
  ... then the next run                 2,200,000           0        0           0   4.4s  PASS
5 run stops AFTER writing: right after  2,200,000           0        0     100,000         PASS
  ... then the next run                 2,300,000           0        0           0   3.5s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2000000,
   'lake': 2000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 111.454807,
   'ok': True,
   'problems': []},
  19.393303632736206),
 ('2 new records arrive, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 7,
   'mb': 112.556471,
   'ok': True,
   'problems': []},
  3.5537757873535156),
 ('3 nothing new, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 7,
   'mb': 112.556471,
   'ok': True,
   'problems': []},
  0.08145332336425781),
 ('4 run stops WHILE writing: right after',
  {'topic': 2200000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 7,
   'mb': 112.556471,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2200000,
   'lake': 2200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 11,
   'mb': 113.710627,
   'ok': True,
   'problems': []}